# Import Libraries

In [1]:
import os
import sqlite3
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import Markdown, display

c:\Users\EduTech\anaconda3\envs\py310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\EduTech\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\hadoop\tmp\ipykernel_20348\2888644238.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github

# Load Gemini API Key

In [3]:
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

print("First 5 characters:", gemini_api_key[:5])

genai.configure(api_key=gemini_api_key)

First 5 characters: AIzaS


# Helper Markdown Function

In [4]:
def print_markdown(text):
    display(Markdown(text))

# =========================================
# 🔵 PART 1 — STATELESS AGENT (No Memory)
# =========================================

# Create Market Research Agent (Stateless)

In [7]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")

market_research_instructions = """
You are a market research assistant helping analyze companies, industries, and competitors.

When given a question:
- Provide a short factual answer.
- Respond in one concise sentence.
"""

# Ask First Question (Q1)

In [8]:
q1 = "What is the market share of Tesla in the US EV market?"

response1 = model.generate_content(
    market_research_instructions + "\n\nQuestion: " + q1
)

print_markdown(f"### Q1: {q1}")
print_markdown("### 🤖 Agent Response:")
print_markdown(response1.text)

### Q1: What is the market share of Tesla in the US EV market?

### 🤖 Agent Response:

Tesla holds approximately 50-60% market share in the US electric vehicle market.

# Ask Follow-up Question (Q2)

In [9]:
q2 = "How does that compare to last year?"

response2 = model.generate_content(
    market_research_instructions + "\n\nQuestion: " + q2
)

print_markdown(f"### Q2: {q2}")
print_markdown("### 🤖 Agent Response:")
print_markdown(response2.text)

### Q2: How does that compare to last year?

### 🤖 Agent Response:

To answer that, please provide the specific data or metric you'd like to compare to last year.

# =========================================
# 🔵 PART 2 — ADD MEMORY USING SQLITE
# ========================================

# Create SQLite Database

In [11]:
conn = sqlite3.connect("conversation.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS conversation (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    role TEXT,
    message TEXT
)
""")

conn.commit()

# Helper Functions for Memory

In [13]:
def save_message(role, message):
    cursor.execute("INSERT INTO conversation (role, message) VALUES (?, ?)", (role, message))
    conn.commit()

def load_conversation():
    cursor.execute("SELECT role, message FROM conversation ORDER BY id ASC")
    return cursor.fetchall()

# Memory-Enabled Agent Function

In [14]:
def run_agent_with_memory(user_input):
    
    # Save user message
    save_message("user", user_input)
    
    # Load entire conversation history
    history = load_conversation()
    
    conversation_text = market_research_instructions + "\n\nConversation History:\n"
    
    for role, message in history:
        conversation_text += f"{role.upper()}: {message}\n"
    
    response = model.generate_content(conversation_text)
    
    # Save assistant response
    save_message("assistant", response.text)
    
    return response.text

# Test With Memory (Q1 Again)

In [15]:
print_markdown("## 🔹 Memory-Enabled Agent")

answer1 = run_agent_with_memory(
    "What is the market share of Tesla in the US EV market?"
)

print_markdown("### 🤖 Agent Response:")
print_markdown(answer1)

## 🔹 Memory-Enabled Agent

### 🤖 Agent Response:

Tesla held approximately 60% of the US EV market share in the first quarter of 2023.

# Ask Follow-Up (Q2)

In [16]:
answer2 = run_agent_with_memory(
    "How does that compare to last year?"
)

print_markdown("### 🤖 Agent Response:")
print_markdown(answer2)

### 🤖 Agent Response:

Tesla's market share in the US EV market has decreased from around 70% in the same period last year.

# PRACTICE OPPORTUNITY SOLUTION:

# Using OpenAI Agents SDK, create a new AI agent named Travel Planner that always suggests one sunny (warm) weekend getaway destination.
# 1. Write the agent’s instructions; tell it to only give one sunny/warm destination.
# 2. Use the latest gpt-5-mini model.
# 3. Save the conversation history with SQLiteSession so it remembers past user questions and doesn’t repeat destinations.
# 4. Ask the agent:
# Please suggest one weekend destination within 5 hours flying from Toronto, Canada
# 5. Display the agent’s answer.
# 6. Then ask a follow-up question about visa requirements for Canadians for that destination and display the answer.
# Hint: Use Runner.run() to send the message to the agent and print the reply.